# 1장 Table 1, 군 비교와 회귀 (R과 통계분석)

MASS 패키지의 출생체중 자료(birthwt)로 Table 1을 만들고, 두 군의 평균과 비율을 비교하고, 선형회귀와 로지스틱 회귀를 맞춥니다. 보건통계학 기초의 파이썬 실습 2·5·9와 같은 자료입니다. Colab에서는 런타임 → 런타임 유형 변경에서 R을 고른 뒤 셀을 위에서부터 차례로 실행하세요.


## a. 자료를 분석할 모양으로 다듬기

### 셀 1. 출생체중 자료 불러오기

In [ ]:
library(MASS)                 # R과 함께 설치되는 패키지. birthwt 자료가 들어 있다
bw <- birthwt
dim(bw)
head(bw, 3)

### 셀 2. 범주형 변수에 이름표와 순서 붙이기

In [ ]:
bw$race <- factor(bw$race, levels = 1:3,
                  labels = c("white", "black", "other"))   # 첫 수준이 기준범주
bw$smoke_f <- factor(bw$smoke, levels = c(0, 1),
                     labels = c("Nonsmoker", "Smoker"))
bw$ptd <- as.integer(bw$ptl > 0)       # 이전 조산 1회 이상 = 1
table(bw$race)
table(bw$smoke_f, bw$smoke)

## b. Table 1 만들기

### 셀 3. CreateTableOne()으로 Table 1 만들기

In [ ]:
install.packages("tableone")
library(tableone)
vars <- c("age", "lwt", "race", "ptd", "ht", "ui", "ftv", "bwt", "low")
t1 <- CreateTableOne(vars = vars, strata = "smoke_f", data = bw,
                     factorVars = c("ptd", "ht", "ui", "low"))
print(t1, smd = TRUE)

### 셀 4. 중앙값과 정확검정으로 바꾸고 파일로 남기기

In [ ]:
tab1 <- print(t1, nonnormal = c("ftv", "bwt"), exact = "ht", smd = TRUE,
              quote = FALSE, noSpaces = TRUE, printToggle = FALSE)
write.csv(tab1, "table1.csv")          # 엑셀에서 열어 논문 표로 옮긴다
noquote(tab1[c("ht = 1 (%)", "bwt (median [IQR])"), ])

## c. 두 군의 평균과 중앙값 비교

### 셀 5. t.test()의 기본값은 Welch

In [ ]:
t.test(bwt ~ smoke_f, data = bw)

### 셀 6. Student t 검정과 나란히 보기

In [ ]:
library(broom)
rbind(tidy(t.test(bwt ~ smoke_f, data = bw)),
      tidy(t.test(bwt ~ smoke_f, data = bw, var.equal = TRUE))
      )[, c("method", "statistic", "parameter", "p.value", "conf.low", "conf.high")]

### 셀 7. 중앙값과 Wilcoxon 순위합 검정

In [ ]:
aggregate(bwt ~ smoke_f, data = bw, FUN = quantile, probs = c(0.25, 0.5, 0.75))
wilcox.test(bwt ~ smoke_f, data = bw)

## d. 두 군의 비율 비교

### 셀 8. 교차표와 행 백분율

In [ ]:
tab <- table(smoke = bw$smoke_f, low = bw$low)
addmargins(tab)
round(100 * prop.table(tab, margin = 1), 1)      # 행 백분율

### 셀 9. chisq.test()의 기본값은 Yates 보정

In [ ]:
chisq.test(tab)
chisq.test(tab, correct = FALSE)
chisq.test(tab)$expected

### 셀 10. Fisher의 정확한 검정과 두 가지 오즈비

In [ ]:
fisher.test(tab)
(tab[2, 2] * tab[1, 1]) / (tab[2, 1] * tab[1, 2])    # 표에서 바로 구한 오즈비

## e. 선형회귀와 로지스틱 회귀

### 셀 11. 선형회귀로 보정한 체중 차이

In [ ]:
fit_lm <- lm(bwt ~ age + lwt + race + smoke + ptd + ht + ui, data = bw)
round(cbind(estimate = coef(fit_lm), confint(fit_lm)), 1)

### 셀 12. glm()으로 로지스틱 회귀

In [ ]:
fit_lr <- glm(low ~ age + lwt + race + smoke + ptd + ht + ui,
              family = binomial, data = bw)
summary(fit_lr)

### 셀 13. 오즈비와 두 가지 신뢰구간

In [ ]:
round(exp(cbind(OR = coef(fit_lr), confint.default(fit_lr))), 3)   # Wald 구간
round(exp(confint(fit_lr)), 3)                                      # 프로파일 우도 구간

### 셀 14. 논문에 옮길 회귀 결과 표

In [ ]:
or_tab <- tidy(fit_lr, exponentiate = TRUE, conf.int = TRUE)[-1, ]   # 절편 행 빼기
data.frame(term = or_tab$term,
           `OR (95% CI)` = sprintf("%.2f (%.2f-%.2f)", or_tab$estimate,
                                   or_tab$conf.low, or_tab$conf.high),
           P = format.pval(or_tab$p.value, digits = 2, eps = 0.001),
           check.names = FALSE)

## f. 과제 정답

### 셀 15. 과제 1 정답: factorVars를 빼면

In [ ]:
t0 <- CreateTableOne(vars = vars, strata = "smoke_f", data = bw)   # factorVars 없이
print(t0)

### 셀 16. 과제 2 정답: 체중 10 lb당 오즈비

In [ ]:
b  <- coef(fit_lr)["lwt"]
ci <- confint.default(fit_lr)["lwt", ]
round(exp(10 * c(OR = b, ci)), 3)        # 10 lb당 오즈비와 Wald 구간

### 셀 17. 과제 3 정답: 보정하지 않은 오즈비

In [ ]:
crude <- glm(low ~ smoke, family = binomial, data = bw)
round(exp(cbind(OR = coef(crude), confint.default(crude))), 3)